# Modular Addition

In [ ]:
from collections import Counter

from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.modular_addition_catalog import modular_addition_examples
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

examples = modular_addition_examples()


def show_signatures(*names):
    display(SVG(field_signature_svg([(name, examples[name].signature, {}) for name in names])))


def show_calls(example):
    graph = example.call_graph()
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)


## `ModAdd`
An n-bit modular addition gate.

Implements |x>|y> => |x>|y + x % p> using $4n$ Toffoli
gates.

This gate can also operate on integers in the Montgomery form.

#### Parameters
 - `bitsize`: Number of bits used to represent each integer.
 - `mod`: The modulus for the addition. 

#### Registers
 - `x`: A bitsize-sized input register (register x above).
 - `y`: A bitsize-sized input/output register (register y above). 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Construction from Figure 6a and cost summary in Figure 8.


In [ ]:
display("ModAdd")

### Example Instances

In [ ]:
mod_add = examples["mod_add"]

#### Graphical Signature

In [ ]:
show_signatures('mod_add')

### Call Graph

In [ ]:
show_calls(mod_add)

## `ModAddK`
Applies U(add, M)|x> = |(x + add) % M> if x < M else |x>.

Applies modular addition to input register `|x>` given parameters `mod` and `add_val` s.t.
 1. If integer `x` < `mod`: output is `|(x + add) % M>`
 2. If integer `x` >= `mod`: output is `|x>`.

This condition is needed to ensure that the mapping of all input basis states (i.e. input
states |0>, |1>, ..., |2 ** bitsize - 1) to corresponding output states is bijective and thus
the gate is reversible.

Also supports controlled version of the gate by specifying a per qubit control value as a tuple
of integers passed as `cvs`.

In [ ]:
display("ModAddK")

### Example Instances

In [ ]:
mod_add_k = examples["mod_add_k"]

In [ ]:
mod_add_k_small = examples["mod_add_k_small"]

In [ ]:
mod_add_k_large = examples["mod_add_k_large"]

#### Graphical Signature

In [ ]:
show_signatures('mod_add_k', 'mod_add_k_small', 'mod_add_k_large')

### Call Graph

In [ ]:
show_calls(mod_add_k)

## `CModAdd`
Controlled Modular Addition.

Implements $\ket{c}\ket{x}\ket{y} \rightarrow \ket{c}\ket{x}\ket{(cx+y)\%p}$
using $5n+1$ Toffoli gates.

Note: The reference reports $5n$ toffolis. Our construction has an extra toffoli gate due
to the current implementaiton of `OutOfPlaceAdder`.

#### Parameters
 - `dtype`: Type of the input registers.
 - `mod`: The modulus for the addition.
 - `cv`: Control value for which the gate is active. 

#### Registers
 - `ctrl`: The control qubit.
 - `x`: A dtype register.
 - `y`: A dtype register. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Construction from Figure 6a and cost summary in Figure 8.


In [ ]:
display("CModAdd")

### Example Instances

In [ ]:
cmodadd_example = examples["cmodadd_example"]

In [ ]:
cmodadd_symbolic = examples["cmodadd_symbolic"]

#### Graphical Signature

In [ ]:
show_signatures('cmodadd_example', 'cmodadd_symbolic')

### Call Graph

In [ ]:
show_calls(cmodadd_example)

## `CModAddK`
Perform x += k mod m for constant k, m and quantum x.

#### Parameters
 - `k`: The integer to add to `x`.
 - `mod`: The modulus for the addition.
 - `bitsize`: The bitsize of the `x` register. 

#### Registers
 - `ctrl`: The control bit
 - `x`: The register to perform the in-place modular addition. 

#### References
 - [How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits](https://arxiv.org/abs/1905.09749). Gidney and Ekerå 2019. The reference implementation in section 2.2 uses CModAddK, but the circuit that it points to is just ModAdd (not ModAddK). This ModAdd is less efficient than the circuit later introduced in the Litinski paper so we choose to use that since it is more efficient and already implemented in Qualtran.
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Litinski et al. 2023. This CModAdd circuit uses 2 fewer additions than the implementation referenced in the paper above. Because of this we choose to use this CModAdd bloq instead.


In [ ]:
display("CModAddK")

### Example Instances

In [ ]:
cmod_add_k = examples["cmod_add_k"]

In [ ]:
cmod_add_k_small = examples["cmod_add_k_small"]

#### Graphical Signature

In [ ]:
show_signatures('cmod_add_k', 'cmod_add_k_small')

### Call Graph

In [ ]:
show_calls(cmod_add_k)

## `CtrlScaleModAdd`
Perform y += x*k mod m for constant k, m and quantum x, y.

#### Parameters
 - `k`: The constant integer to scale `x` before adding into `y`.
 - `mod`: The modulus of the addition
 - `bitsize`: The size of the two registers. 

#### Registers
 - `ctrl`: The control bit
 - `x`: The 'source' quantum register containing the integer to be scaled and added to `y`.
 - `y`: The 'destination' quantum register to which the addition will apply. 

#### References
 - [How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits](https://arxiv.org/abs/1905.09749). Construction based on description in section 2.2 paragraph 4. We add n And/And† bloqs because the bloq is controlled, but the construction also involves modular addition controlled on the qubits comprising register x.


In [ ]:
display("CtrlScaleModAdd")

### Example Instances

In [ ]:
ctrl_scale_mod_add = examples["ctrl_scale_mod_add"]

In [ ]:
ctrl_scale_mod_add_small = examples["ctrl_scale_mod_add_small"]

#### Graphical Signature

In [ ]:
show_signatures('ctrl_scale_mod_add', 'ctrl_scale_mod_add_small')

### Call Graph

In [ ]:
show_calls(ctrl_scale_mod_add)

### Native execution

The bounded constant example preserves all binary padding states. Its native gate counts come from the emitted circuit. The original one-level graph above declares five Add calls.
Both modular-addition paths require caller-owned clean work.

In [ ]:
for example in (mod_add_k_small, cmod_add_k_small, ctrl_scale_mod_add_small):
    workspace, slots = example.workspace_requirements()
    width = int(example.signature.n_qubits()) + workspace
    circuit = DynamicCircuit(width).append(example)
    display((example.kind, workspace, slots))
    display(Counter(gate.name for gate in circuit.to_ir().instructions))